In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device.upper()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

In [ ]:
!git clone https://github.com/Enitity505/HaluEval.git

Cloning into 'HaluEval'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 171 (delta 5), reused 2 (delta 2), pack-reused 141 (from 2)
Receiving objects: 100% (171/171), 39.21 MiB | 17.17 MiB/s, done.
Resolving deltas: 100% (76/76), done.
Updating files: 100% (38/38), done.


In [ ]:
print("\n dialogue data \n")
!head HaluEval/data/dialogue_data.json
print("\n genral data \n")
!head HaluEval/data/general_data.json
print("\n qa data \n")
!head HaluEval/data/qa_data.json
print("\n summarization data \n")
!head HaluEval/data/summarization_data.json





 dialogue data 

{"knowledge": "Iron Man is starring Robert Downey Jr.Robert Downey Jr. starred in Zodiac (Crime Fiction Film)Zodiac (Crime Fiction Film) is starring Jake Gyllenhaal", "dialogue_history": "[Human]: Do you like Iron Man [Assistant]: Sure do! Robert Downey Jr. is a favorite. [Human]: Yes i like him too did you know he also was in Zodiac a crime fiction film. ", "right_response": "I like crime fiction! Didn't know RDJ was in there. Jake Gyllenhaal starred as well.", "hallucinated_response": "I'm not a fan of crime movies, but I did know that RDJ starred in Zodiac with Tom Hanks."}
{"knowledge": "The Secret Life of Bees has genre Teen drama. Teen drama has examples: A Walk to RememberA Walk to Remember is produced by Hunt LowryHunt Lowry produced A Walk to Remember. A Walk to Remember is a/an Film", "dialogue_history": "[Human]: Could you recommend any books like The Secret Life of Bees? [Assistant]: Yes, its a great teen book, have you heard about A walk to remember ? It 

In [ ]:
import json
import traceback
def file_loader(filepath):
    try:
        data = []
        with open(filepath, 'r', encoding='utf-8') as f:
            for i, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                    data.append(obj)
                except json.JSONDecodeError as je:
                    print(f"JSON decode error on line {i+1}: {je}")
                    print(f"Problematic line (first 100 chars): {line[:100]}")
                    continue

        print(f"Loaded {len(data)} samples from {filepath}")
        return data
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        print("Please ensure the data file exists in the current directory.")
        return []
    except Exception as e:
        print(f"Error loading data: {e}")
        traceback.print_exc()
        return []

dialogue_data=file_loader("HaluEval/data/dialogue_data.json")
general_data=file_loader("HaluEval/data/general_data.json")
qa_data=file_loader("HaluEval/data/qa_data.json")
summarization_data=file_loader("HaluEval/data/summarization_data.json")






Loaded 10000 samples from HaluEval/data/dialogue_data.json
Loaded 4507 samples from HaluEval/data/general_data.json
Loaded 10000 samples from HaluEval/data/qa_data.json
Loaded 10000 samples from HaluEval/data/summarization_data.json


In [ ]:
import pandas as pd

def process_paired_data(data, context_key, prompt_key, right_key, wrong_key, task_type, default_prompt=None):
    records = []
    for idx, item in enumerate(data):
        prompt = default_prompt if default_prompt else item.get(prompt_key, "")
        # Both variants of one example share a group_id. Splitting must keep them
        # together, otherwise the same context leaks across train/test and the
        # reported scores become optimistic.
        group_id = f"{task_type}_{idx}"

        # Factual Record
        records.append({
            "context": item.get(context_key, ""),
            "prompt": prompt,
            "response": item.get(right_key, ""),
            "is_hallucination": 0,
            "task_type": task_type,
            "group_id": group_id
        })
        # Hallucinated Record
        records.append({
            "context": item.get(context_key, ""),
            "prompt": prompt,
            "response": item.get(wrong_key, ""),
            "is_hallucination": 1,
            "task_type": task_type,
            "group_id": group_id
        })
    return records

def process_general_data(data):
    records = []
    for idx, item in enumerate(data):
        is_hallucinated = 1 if item.get("hallucination", "").lower() == "yes" else 0
        records.append({
            "context": "",
            "prompt": item.get("user_query", ""),
            "response": item.get("chatgpt_response", ""),
            "is_hallucination": is_hallucinated,
            "task_type": "general",
            "group_id": f"general_{idx}"
        })
    return records

# Consolidate all records into the required triplet (Context, Prompt, Answer)
all_records = []
if dialogue_data:
    all_records.extend(process_paired_data(dialogue_data, "knowledge", "dialogue_history", "right_response", "hallucinated_response", "dialogue"))
if qa_data:
    all_records.extend(process_paired_data(qa_data, "knowledge", "question", "right_answer", "hallucinated_answer", "qa"))
if summarization_data:
    all_records.extend(process_paired_data(summarization_data, "document", None, "right_summary", "hallucinated_summary", "summarization", "Summarize the document."))
if general_data:
    all_records.extend(process_general_data(general_data))

# Create tabular schema
df = pd.DataFrame(all_records)
print(f"Data Integration Complete. Unified Schema Created: {df.shape[0]} total rows.")

Data Integration Complete. Unified Schema Created: 64507 total rows.


In [ ]:
!pip install sentence-transformers

In [ ]:
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
import numpy as np
import re

print("Ensuring Text Normalization (Data Cleaning)...")

TOKEN_RE = re.compile(r"\b\w+\b")
NUMBER_RE = re.compile(r"\b\d+\b")


def normalize_text(text):
    """Lowercase + punctuation-stripped normalization.

    IMPORTANT: this must stay in sync with CascadedGuardrailPipeline._normalize,
    otherwise training-time features and inference-time features drift apart
    (train/serve skew).
    """
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text, flags=re.UNICODE)
    return text


def lexical_overlap(reference, response):
    """Jaccard overlap between reference and response vocabularies."""
    ref_tokens = set(TOKEN_RE.findall(reference))
    res_tokens = set(TOKEN_RE.findall(response))
    if not ref_tokens or not res_tokens:
        return 0.0
    return len(ref_tokens & res_tokens) / len(ref_tokens | res_tokens)


def unsupported_number_ratio(reference, response):
    """Share of numeric tokens in the response that never appear in the reference.

    Cheap but strong signal for the "invented figures / dates / quantities"
    hallucination failure mode that this project targets.
    """
    ref_numbers = set(NUMBER_RE.findall(reference))
    resp_numbers = NUMBER_RE.findall(response)
    if not resp_numbers:
        return 0.0
    unsupported = sum(1 for number in resp_numbers if number not in ref_numbers)
    return unsupported / len(resp_numbers)


# Safely create the _clean columns if they don't exist
if 'response_clean' not in df.columns:
    df['context_clean'] = df['context'].apply(normalize_text)
    df['prompt_clean'] = df['prompt'].apply(normalize_text)
    df['response_clean'] = df['response'].apply(normalize_text)


print("Step 4: Transformation - Selecting References & Validating...")

# 1. Validate against empty responses before proceeding
if df['response_clean'].fillna("").str.strip().eq("").any():
    print("Warning: Empty responses found. Dropping them now.")
    df = df[df['response_clean'].fillna("").str.strip() != ""]

# 2. Clean nulls before string conversion to prevent the 'str(np.nan) -> "nan"' bug
context_clean = df['context_clean'].fillna('').astype(str)
prompt_clean = df['prompt_clean'].fillna('').astype(str)
responses = df['response_clean'].fillna('').astype(str)

# 3. Vectorized fallback: context_missing flag and reference_text selection
df['context_missing'] = np.where(context_clean.str.strip() == "", 1, 0)
df['reference_text'] = np.where(
    df['context_missing'] == 1,
    prompt_clean,
    context_clean
)

print("Step 5: Feature Engineering - Calculating Word Counts & Length Ratios...")

# 4. Calculate whitespace-separated word counts using vectorized pandas string methods
df['reference_length'] = df['reference_text'].str.split().str.len().fillna(0).astype(int)
df['response_length'] = responses.str.split().str.len().fillna(0).astype(int)

# 5. Calculate Length Ratio (response / reference) with vectorized zero-division protection
df['length_ratio'] = np.divide(
    df['response_length'],
    df['reference_length'],
    out=np.zeros(len(df), dtype=float),
    where=df['reference_length'].to_numpy() > 0
)

print("Step 5b: Feature Engineering - Lexical Overlap & Unsupported Numbers...")

# 6. Lexical (surface-form) overlap between reference and response
df['lexical_overlap'] = [
    lexical_overlap(ref, res) for ref, res in zip(df['reference_text'], responses)
]

# 7. Fraction of numbers in the response that are absent from the reference
df['unsupported_number_ratio'] = [
    unsupported_number_ratio(ref, res) for ref, res in zip(df['reference_text'], responses)
]

print("Loading Sentence Transformer model on GPU...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# 8. Generate Embeddings (Reference, Response and Prompt)
reference_texts = df['reference_text'].tolist()
responses_list = responses.tolist()
prompts_list = prompt_clean.tolist()

print(f"Generating embeddings using PyTorch on {device}...")
ref_embeddings = model.encode(reference_texts, batch_size=64, convert_to_tensor=True, show_progress_bar=True)
resp_embeddings = model.encode(responses_list, batch_size=64, convert_to_tensor=True, show_progress_bar=True)
prompt_embeddings = model.encode(prompts_list, batch_size=64, convert_to_tensor=True, show_progress_bar=True)

print("Calculating Reference/Response and Prompt/Response Semantic Similarity...")
# 9. Cosine similarities - the semantic core of the feature set
df['cosine_similarity'] = F.cosine_similarity(ref_embeddings, resp_embeddings, eps=1e-8).cpu().numpy()
df['prompt_response_cosine'] = F.cosine_similarity(prompt_embeddings, resp_embeddings, eps=1e-8).cpu().numpy()

# 10. The embedding tensors are no longer needed - free the memory
del ref_embeddings, resp_embeddings, prompt_embeddings
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 11. Clean Numerical Interface Contract
FEATURE_COLUMNS = [
    "cosine_similarity",
    "prompt_response_cosine",
    "reference_length",
    "response_length",
    "length_ratio",
    "lexical_overlap",
    "unsupported_number_ratio",
    "context_missing"
]
TARGET_COLUMN = "is_hallucination"

# Validate target column integrity
if df[TARGET_COLUMN].isnull().any():
    raise ValueError("Missing target labels detected. Do not impute the target.")

# Create the final engineered datasets for Features (X) and Target (y)
X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

# Final validation for NaN/Inf values in features only
X.replace([np.inf, -np.inf], np.nan, inplace=True)
if X.isnull().values.any():
    print("Warning: NaN values detected in feature set. Imputing with 0 as a baseline fallback.")
    X.fillna(0, inplace=True)

print("Running Sanity Checks on Numerical Interface...")
assert X["cosine_similarity"].between(-1.0001, 1.0001).all(), "Cosine similarity out of bounds."
assert X["prompt_response_cosine"].between(-1.0001, 1.0001).all(), "Prompt-response cosine out of bounds."
assert (X["reference_length"] >= 0).all(), "Negative reference lengths found."
assert (X["response_length"] >= 0).all(), "Negative response lengths found."
assert (X["length_ratio"] >= 0).all(), "Negative length ratios found."
assert X["lexical_overlap"].between(0, 1).all(), "lexical_overlap out of [0, 1]."
assert X["unsupported_number_ratio"].between(0, 1).all(), "unsupported_number_ratio out of [0, 1]."
assert X["context_missing"].isin([0, 1]).all(), "context_missing contains values other than 0 or 1."

print("\nPipeline Complete! Engineered Feature Dataset (X) ready for scaling and training.")
display(X.head())

In [ ]:
import os
import sys
import json
import joblib
import torch
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import average_precision_score, precision_recall_curve
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Safety fallbacks in case earlier setup cells were skipped or runtime restarted
RANDOM_SEED = globals().get('RANDOM_SEED', 42)
MIN_PRECISION = globals().get('MIN_PRECISION', 0.75)

print("--- Step 6 & 7: Multi-Model Training, Gated Tier-2 Score Tuning & Serialization ---")

--- Step 6 & 7: Multi-Model Training, Gated Tier-2 Score Tuning & Serialization ---


In [ ]:
# =====================================================================
# ANN: PyTorch multi-layer perceptron with a scikit-learn compatible API
# =====================================================================
# Why a wrapper? It lets the ANN plug straight into the existing soft-voting
# ensemble, the grouped-OOF threshold tuning, joblib serialization and the
# cascaded guardrail without special-casing any downstream code.
#
# NOTE: joblib pickles this class by reference (__main__.TorchMLPClassifier),
# so this cell must be executed before loading saved artifacts in a fresh
# runtime.

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import average_precision_score
from sklearn.model_selection import train_test_split as _tts


class _HallucinationMLP(nn.Module):
    """Small feed-forward network for tabular hallucination risk scoring."""

    def __init__(self, input_dim, hidden_dims=(128, 64), dropout=0.2):
        super().__init__()
        layers, prev = [], input_dim
        for hidden in hidden_dims:
            layers += [
                nn.Linear(prev, hidden),
                nn.LayerNorm(hidden),   # LayerNorm, not BatchNorm, so batch size 1 never breaks
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            prev = hidden
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).view(-1)


class TorchMLPClassifier(ClassifierMixin, BaseEstimator):
    """PyTorch MLP exposed with the sklearn estimator API.

    Design notes
    ------------
    * Class imbalance is handled with BCEWithLogitsLoss(pos_weight=neg/pos),
      mirroring class_weight='balanced' on the other Tier-1 models.
    * Early stopping tracks validation average precision (the project's main
      metric) and restores the best epoch.
    * Inputs are expected to be already scaled (the notebook applies a
      StandardScaler before the ensemble).
    """

    def __init__(self, hidden_dims=(128, 64), dropout=0.2, epochs=30,
                 batch_size=256, lr=1e-3, weight_decay=1e-4, patience=5,
                 val_fraction=0.1, device=None, random_state=42, verbose=False):
        # sklearn convention: __init__ must only store the parameters verbatim
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.patience = patience
        self.val_fraction = val_fraction
        self.device = device
        self.random_state = random_state
        self.verbose = verbose

    # ------------------------------------------------------------------
    # internals
    # ------------------------------------------------------------------
    def _resolve_device(self):
        if self.device:
            return torch.device(self.device)
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _loader(self, X, y, shuffle):
        X_t = torch.from_numpy(np.asarray(X, dtype=np.float32))
        y_t = torch.from_numpy(np.asarray(y, dtype=np.float32))
        return DataLoader(
            TensorDataset(X_t, y_t),
            batch_size=self.batch_size,
            shuffle=shuffle,
            drop_last=False,
        )

    def _forward_logits(self, X):
        self.model_.eval()
        # Inference follows whatever device the module currently lives on.
        # After fit() the module is parked on CPU so joblib artifacts stay
        # portable (no CUDA tensors inside the pickled ensemble).
        device = next(self.model_.parameters()).device
        chunks = []
        with torch.inference_mode():
            for xb, _ in self._loader(X, np.zeros(len(X)), shuffle=False):
                chunks.append(self.model_(xb.to(device)).cpu())
        return torch.cat(chunks)

    # ------------------------------------------------------------------
    # sklearn API
    # ------------------------------------------------------------------
    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).ravel()

        self.classes_ = np.unique(y)
        if self.classes_.size < 2:
            raise ValueError("TorchMLPClassifier needs at least two classes to train.")
        self.n_features_in_ = X.shape[1]

        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)
        self.device_ = self._resolve_device()

        # Stratified inner split used only for early stopping / epoch selection.
        X_tr, X_val, y_tr, y_val = _tts(
            X, y,
            test_size=self.val_fraction,
            random_state=self.random_state,
            stratify=y,
        )

        self.model_ = _HallucinationMLP(
            input_dim=X.shape[1],
            hidden_dims=tuple(self.hidden_dims),
            dropout=self.dropout,
        ).to(self.device_)

        positives = float(y_tr.sum())
        negatives = float(len(y_tr) - positives)
        pos_weight = torch.tensor([negatives / max(positives, 1.0)],
                                  dtype=torch.float32, device=self.device_)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(
            self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay
        )
        train_loader = self._loader(X_tr, y_tr, shuffle=True)

        best_ap, best_state, bad_epochs, epoch = -1.0, None, 0, 0
        for epoch in range(1, self.epochs + 1):
            self.model_.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(self.device_), yb.to(self.device_)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(self.model_(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model_.parameters(), 5.0)
                optimizer.step()

            val_probs = torch.sigmoid(self._forward_logits(X_val)).numpy()
            if np.unique(y_val).size < 2:
                val_ap = 0.0
            else:
                val_ap = float(average_precision_score(y_val, val_probs))

            if val_ap > best_ap + 1e-5:
                best_ap = val_ap
                best_state = {k: v.detach().cpu().clone()
                              for k, v in self.model_.state_dict().items()}
                bad_epochs = 0
            else:
                bad_epochs += 1

            if self.verbose and (epoch == 1 or epoch % 5 == 0):
                print(f"    [ANN] epoch {epoch:02d} | val AP {val_ap:.4f} | best {best_ap:.4f}")

            if bad_epochs >= self.patience:
                break

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.model_.to("cpu")  # keep the serialized artifact device-agnostic
        self.best_val_ap_ = float(best_ap)
        self.n_epochs_trained_ = epoch
        return self

    def predict_proba(self, X):
        if not hasattr(self, "model_"):
            raise RuntimeError("TorchMLPClassifier is not fitted yet. Call fit() first.")
        probs = torch.sigmoid(self._forward_logits(np.asarray(X, dtype=np.float32))).numpy()
        return np.column_stack([1.0 - probs, probs])

    def predict(self, X):
        return self.classes_[(self.predict_proba(X)[:, 1] >= 0.5).astype(int)]

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

print("Splitting data into train and test sets (group-disjoint + stratified)...")
# Keep the (right, hallucinated) twins of each HaluEval example together so the
# same context can never appear in both splits. Same seed/strategy as the
# training cell, so running either order produces the identical split.
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(
    outer_splitter.split(df[FEATURE_COLUMNS], y, groups=df['group_id'])
)

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

# Map the correctly split variables for the pipeline
X_train_raw = df_train[FEATURE_COLUMNS].values
y_train_full = y_train.to_numpy() if hasattr(y_train, 'to_numpy') else np.array(y_train)
groups_train = df_train['group_id'].to_numpy()
train_strat_key = y_train_full


In [ ]:
import os
import sys
import json
import joblib
import torch
import numpy as np
import pandas as pd
import sklearn
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import average_precision_score, precision_recall_curve
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Safety fallbacks
RANDOM_SEED = 42
MIN_PRECISION = 0.75

print("--- Step 6 & 7: Multi-Model Training, Gated Tier-2 Score Tuning & Serialization ---")

print("Splitting data into train and test sets (group-disjoint + stratified)...")
# 1. Strict 80/20 split that keeps (right, hallucinated) twins of the same
#    HaluEval example on the same side of the boundary. Using a single fold of
#    StratifiedGroupKFold gives a stratified, group-disjoint 20% hold-out.
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
train_idx, test_idx = next(
    outer_splitter.split(df[FEATURE_COLUMNS], y, groups=df['group_id'])
)

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

# 2. Map the correctly split variables for the training pipeline
X_train_raw = df_train[FEATURE_COLUMNS].values
y_train_full = y_train.to_numpy() if hasattr(y_train, 'to_numpy') else np.array(y_train)
groups_train = df_train['group_id'].to_numpy()
train_strat_key = y_train_full

# 3. Define Tier-1 Base Estimators
if 'TorchMLPClassifier' not in globals():
    raise RuntimeError("TorchMLPClassifier is not defined. Run the ANN definition cell first.")

tier1_estimators = [
    ("lr", LogisticRegression(class_weight='balanced', random_state=RANDOM_SEED, max_iter=1000)),
    ("rf", RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)),
    ("dt", DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_SEED)),
    ("svm", SVC(probability=True, class_weight='balanced', random_state=RANDOM_SEED)),
    # PyTorch ANN (multi-layer perceptron) over the engineered feature vector
    ("ann", TorchMLPClassifier(
        hidden_dims=(128, 64),
        dropout=0.2,
        epochs=30,
        batch_size=256,
        lr=1e-3,
        weight_decay=1e-4,
        patience=5,
        random_state=RANDOM_SEED,
    )),
]

# 4. Grouped OOF Evaluation for Tier-1 Voting Ensemble
sk_w_oof_scores = np.zeros(len(y_train_full))
sgkf_inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

for fold_id, (tr_i, va_i) in enumerate(
    sgkf_inner.split(X_train_raw, train_strat_key, groups=groups_train),
    start=1
):
    y_tr = y_train_full[tr_i]
    y_va = y_train_full[va_i]

    if np.unique(y_tr).size < 2:
        raise ValueError(f"Inner fold {fold_id}: training split contains only one class.")
    if np.unique(y_va).size < 2:
        raise ValueError(f"Inner fold {fold_id}: validation split contains only one class.")

    fold_scaler = StandardScaler()
    X_tr = fold_scaler.fit_transform(X_train_raw[tr_i])
    X_va = fold_scaler.transform(X_train_raw[va_i])

    fold_estimators = [(name, clone(estimator)) for name, estimator in tier1_estimators]
    fold_ensemble = VotingClassifier(estimators=fold_estimators, voting='soft')

    fold_ensemble.fit(X_tr, y_tr)
    sk_w_oof_scores[va_i] = fold_ensemble.predict_proba(X_va)[:, 1]

# 5. Define Cascade Hyperparameters (Fixed baseline configuration)
gate_low, gate_high = 0.35, 0.65
tier1_weight, tier2_weight = 0.4, 0.6

# 6. Rigorous Alignment Guarantees
if X_train_raw.shape[0] != len(df_train):
    raise AssertionError(f"Alignment mismatch: X_train_raw has {X_train_raw.shape[0]} rows, but df_train has {len(df_train)} rows.")
if len(y_train_full) != len(df_train):
    raise AssertionError(f"Alignment mismatch: y_train_full has {len(y_train_full)} rows, but df_train has {len(df_train)} rows.")

# 7. Generate Genuine Tier-2 Pretrained Scores across Training Split
print("Generating Tier-2 pretrained scores using cross-encoder/nli-deberta-v3-small...")
tier2_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tier2_name = "cross-encoder/nli-deberta-v3-small"
nli_tokenizer = AutoTokenizer.from_pretrained(tier2_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(tier2_name).to(tier2_device)
nli_model.eval()

nli_id2label = {int(k): v.lower() for k, v in nli_model.config.id2label.items()}
entail_idx = None
for idx, label in nli_id2label.items():
    if "entail" in label:
        entail_idx = idx
        break
if entail_idx is None:
    raise RuntimeError(f"Could not identify entailment label in NLI model config: {nli_id2label}")

train_samples = df_train[['context', 'prompt', 'response']].to_dict('records')
tier2_train_scores = np.zeros(len(y_train_full))

batch_size = 32
for i in range(0, len(train_samples), batch_size):
    batch = train_samples[i:i+batch_size]
    premises = [
        row['prompt'] if (pd.isna(row['context']) or str(row['context']).strip() == "") else row['context']
        for row in batch
    ]
    hypotheses = [row['response'] for row in batch]

    inputs = nli_tokenizer(premises, hypotheses, return_tensors="pt", truncation=True, max_length=512, padding=True).to(tier2_device)
    with torch.inference_mode():
        outputs = nli_model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        entail_probs = probs[:, entail_idx].cpu().numpy()
        hallucination_scores = 1.0 - entail_probs

    tier2_train_scores[i:i+batch_size] = hallucination_scores

# 8. Score Sanity Checks
if not np.isfinite(sk_w_oof_scores).all():
    raise ValueError("Tier-1 OOF scores contain NaN or Inf.")
if not np.isfinite(tier2_train_scores).all():
    raise ValueError("Tier-2 scores contain NaN or Inf.")
if not (np.all((sk_w_oof_scores >= 0) & (sk_w_oof_scores <= 1)) and np.all((tier2_train_scores >= 0) & (tier2_train_scores <= 1))):
    raise ValueError("Invalid probability/risk score range detected.")

# 9. Threshold Tuning Helpers & Gated Selection
def select_threshold(y_true, y_scores, name):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_scores)
    valid_mask = precisions[:-1] >= MIN_PRECISION

    # Graceful Fallback: If 75% precision is impossible on this specific subset, maximize F1 instead.
    if not valid_mask.any():
        print(f"\n⚠️ WARNING: '{name}' cannot achieve {MIN_PRECISION:.0%} precision on its specific subset.")
        print("   -> Falling back to the threshold that maximizes F1 score for these edge cases.")
        f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-9)
        optimal_idx = np.argmax(f1_scores)
        return thresholds[optimal_idx]

    optimal_idx = np.argmax(recalls[:-1][valid_mask])
    return thresholds[valid_mask][optimal_idx]

tier1_threshold = select_threshold(y_train_full, sk_w_oof_scores, "Tier-1 Multi-Model Ensemble")

cascade_validation_scores = (tier1_weight * sk_w_oof_scores) + (tier2_weight * tier2_train_scores)
cascade_mask = (sk_w_oof_scores >= gate_low) & (sk_w_oof_scores <= gate_high)

cascade_count = int(cascade_mask.sum())
cascade_rate = cascade_count / len(cascade_mask)
print(f"Tier-2 gate coverage: {cascade_rate:.1%} ({cascade_count}/{len(cascade_mask)})")

if cascade_count < 20:
    raise RuntimeError(f"Only {cascade_count} samples enter the Tier-2 gate; insufficient data for cascade threshold tuning.")

cascade_threshold = select_threshold(
    y_train_full[cascade_mask],
    cascade_validation_scores[cascade_mask],
    "Cascaded Ensemble on Tier-2 Gated Samples"
)

print(f"\nTier-1 Ensemble OOF AP           : {average_precision_score(y_train_full, sk_w_oof_scores):.3f}")
print(f"Tier-1 Operating Threshold       : {tier1_threshold:.3f}")
print(f"Separately Tuned Cascade Threshold: {cascade_threshold:.3f}")

# 10. Fit Final Tier-1 Ensemble on Complete Training Split
final_scaler = StandardScaler()
X_train_final = final_scaler.fit_transform(X_train_raw)

final_estimators = [(name, clone(estimator)) for name, estimator in tier1_estimators]
tier1_final_model = VotingClassifier(estimators=final_estimators, voting='soft')
tier1_final_model.fit(X_train_final, y_train_full)

# 11. Serialization Setup
ARTIFACT_DIR = "./guardrail_artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

metadata_path = os.path.join(ARTIFACT_DIR, "guardrail_metadata.json")
scaler_path = os.path.join(ARTIFACT_DIR, "guardrail_scaler.joblib")
ensemble_path = os.path.join(ARTIFACT_DIR, "tier1_voting_ensemble.joblib")

deployment_metadata = {
    "artifact_schema_version": "1.1",
    "feature_schema_version": "2.0",
    "model_type": "Multi-Model Voting Ensemble (LR, RF, DT, SVM, PyTorch ANN) + NLI Tier-2 Cascaded Guardrail",
    "threshold": float(tier1_threshold),
    "cascade_threshold": float(cascade_threshold),
    "min_precision": float(MIN_PRECISION),
    "feature_columns": FEATURE_COLUMNS,
    "embedding_config": {
        "model_name": "all-MiniLM-L6-v2",
        "model_revision": None,
        "embedding_dimension": 384,
        "similarity_metric": "cosine_similarity",
        "similarity_eps": 1e-8,
        "normalize_embeddings": False
    },
    "tier1_config": {
        "models": [
            "LogisticRegression",
            "RandomForestClassifier",
            "DecisionTreeClassifier",
            "SVC",
            "TorchMLPClassifier (ANN)"
        ],
        "voting": "soft",
        "n_estimators_rf": 100,
        "class_weight": "balanced"
    },
    "ann_config": {
        "framework": "PyTorch",
        "architecture": "MLP(8 -> 128 -> 64 -> 1), LayerNorm + ReLU + Dropout(0.2)",
        "loss": "BCEWithLogitsLoss(pos_weight=neg/pos)",
        "optimizer": "AdamW",
        "epochs_max": 30,
        "early_stopping_metric": "validation average precision",
        "patience": 5,
        "batch_size": 256
    },
    "tier2_config": {
        "enabled": True,
        "model_name": tier2_name,
        "model_revision": None,
        "loading_strategy": "from_pretrained"
    },
    "cascade_config": {
        "gate_low": gate_low,
        "gate_high": gate_high,
        "tier1_weight": tier1_weight,
        "tier2_weight": tier2_weight
    },
    "software_versions": {
        "python": sys.version.split(' ')[0],
        "torch": torch.__version__,
        "sklearn": sklearn.__version__
    }
}

with open(metadata_path, 'w') as f:
    json.dump(deployment_metadata, f, indent=4)

joblib.dump(final_scaler, scaler_path)
joblib.dump(tier1_final_model, ensemble_path)

print(f"\nSerialization Successful:")
print(f" - Metadata JSON : {metadata_path}")
print(f" - Scaler Joblib : {scaler_path}")
print(f" - Ensemble Joblib: {ensemble_path}")

In [ ]:
# =====================================================================
# Tier-1 diagnostics: per-model AP comparison (incl. the new ANN)
# =====================================================================
import json
import time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score, classification_report, confusion_matrix
)

print("--- Tier-1 Diagnostic: ANN vs. base learners (held-out test set) ---")

_required = ("tier1_final_model", "final_scaler", "df_test", "FEATURE_COLUMNS", "TARGET_COLUMN")
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the feature-engineering and training cells first (missing: {_missing}).")

X_test_scaled = final_scaler.transform(df_test[FEATURE_COLUMNS].values)
y_test_arr = np.asarray(df_test[TARGET_COLUMN]).ravel()

rows = []
for name, estimator in tier1_final_model.named_estimators_.items():
    t0 = time.perf_counter()
    probs = estimator.predict_proba(X_test_scaled)[:, 1]
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    rows.append({
        "model": name.upper(),
        "average_precision": average_precision_score(y_test_arr, probs),
        "inference_ms_total": elapsed_ms,
    })

t0 = time.perf_counter()
ensemble_probs = tier1_final_model.predict_proba(X_test_scaled)[:, 1]
ensemble_ms = (time.perf_counter() - t0) * 1000.0
rows.append({
    "model": "ENSEMBLE",
    "average_precision": average_precision_score(y_test_arr, ensemble_probs),
    "inference_ms_total": ensemble_ms,
})

diagnostic_df = (
    pd.DataFrame(rows)
    .sort_values("average_precision", ascending=False)
    .reset_index(drop=True)
)
diagnostic_df["inference_ms_per_sample"] = diagnostic_df["inference_ms_total"] / len(y_test_arr)
print("\nTier-1 model comparison on the held-out test set:")
display(diagnostic_df.round(4))

# Standalone ANN report at the deployed Tier-1 operating threshold
ann_model = tier1_final_model.named_estimators_["ann"]
ann_probs = ann_model.predict_proba(X_test_scaled)[:, 1]

ann_threshold = 0.5
if "metadata_path" in globals():
    try:
        with open(metadata_path, "r") as f:
            ann_threshold = float(json.load(f)["threshold"])
    except Exception:
        pass

ann_preds = (ann_probs >= ann_threshold).astype(int)
print(f"\nANN standalone report (Tier-1 threshold = {ann_threshold:.3f}):")
print(classification_report(
    y_test_arr, ann_preds,
    target_names=["Safe (0)", "Hallucination (1)"], digits=4
))
cm = confusion_matrix(y_test_arr, ann_preds)
print(f"TN={cm[0][0]}  FP={cm[0][1]}  FN={cm[1][0]}  TP={cm[1][1]}")
print(f"ANN best validation AP observed during training: {getattr(ann_model, 'best_val_ap_', float('nan')):.4f}")

In [ ]:
import os
import re
import json
import joblib
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sentence_transformers import SentenceTransformer


class CascadedGuardrailPipeline:
    """Two-tier hallucination guardrail (single source of truth).

    Tier-1 : StandardScaler -> soft-voting ensemble (LR, RF, DT, SVM, PyTorch ANN)
    Tier-2 : cross-encoder NLI fallback, only for uncertain Tier-1 scores
    """

    EXPECTED_SCHEMA_VERSION = "1.1"

    def __init__(self, ensemble_file, scaler_file, metadata_file, device_str="cpu"):
        self.device = torch.device(device_str)

        with open(metadata_file, 'r') as f:
            meta = json.load(f)

        schema = meta.get("artifact_schema_version")
        if schema != self.EXPECTED_SCHEMA_VERSION:
            raise ValueError(
                f"Artifact schema {schema!r} is incompatible with this pipeline "
                f"(expected {self.EXPECTED_SCHEMA_VERSION!r}). Re-run the feature-engineering "
                "and training cells to regenerate ./guardrail_artifacts."
            )

        if 'threshold' not in meta or 'cascade_threshold' not in meta:
            raise ValueError("Missing 'threshold' or 'cascade_threshold' in metadata.")

        self.threshold = float(meta['threshold'])
        self.cascade_threshold = float(meta['cascade_threshold'])

        cascade_cfg = meta.get('cascade_config', {})
        self.gate_low = float(cascade_cfg.get('gate_low', 0.35))
        self.gate_high = float(cascade_cfg.get('gate_high', 0.65))
        self.tier1_weight = float(cascade_cfg.get('tier1_weight', 0.4))
        self.tier2_weight = float(cascade_cfg.get('tier2_weight', 0.6))

        self.features = meta['feature_columns']
        self.eps = meta['embedding_config']['similarity_eps']

        # Tier-1 assets
        self.scaler = joblib.load(scaler_file)
        self.tier1_model = joblib.load(ensemble_file)
        self.embedder = SentenceTransformer(meta['embedding_config']['model_name'], device=self.device)

        # Tier-2 NLI verifier
        tier2_name = meta['tier2_config']['model_name']
        print(f"Initializing Tier-2 NLI Verifier ({tier2_name}) on {device_str.upper()}...")
        self.tokenizer = AutoTokenizer.from_pretrained(tier2_name)
        self.tier2_model = AutoModelForSequenceClassification.from_pretrained(tier2_name).to(self.device)
        self.tier2_model.eval()

        self.id2label = {int(k): v.lower() for k, v in self.tier2_model.config.id2label.items()}

    # ------------------------------------------------------------------
    # Feature extraction (must mirror the training-time feature cell)
    # ------------------------------------------------------------------
    def _normalize(self, text):
        if pd.isna(text):
            return "missing"
        text = str(text).lower().strip()
        text = re.sub(r'\s+', ' ', text)
        text = re.sub(r'[^\w\s]', '', text, flags=re.UNICODE)
        return text if text else "empty"

    def prepare_text(self, context, prompt, response):
        """Stage 1: normalization + reference fallback resolution."""
        ctx_clean = self._normalize(context)
        prm_clean = self._normalize(prompt)
        res_clean = self._normalize(response)
        context_missing = int(ctx_clean == "empty" or ctx_clean == "missing")
        ref_text = prm_clean if context_missing else ctx_clean
        return ref_text, res_clean, context_missing

    def compute_embeddings(self, ref_text, res_clean, prompt_text):
        """Stage 2: reference/response and prompt/response cosine similarity."""
        embeddings = self.embedder.encode(
            [ref_text, res_clean, prompt_text],
            convert_to_tensor=True,
            show_progress_bar=False
        )
        cos_sim = torch.nn.functional.cosine_similarity(
            embeddings[0].unsqueeze(0),
            embeddings[1].unsqueeze(0),
            eps=self.eps
        ).item()
        prompt_cos = torch.nn.functional.cosine_similarity(
            embeddings[2].unsqueeze(0),
            embeddings[1].unsqueeze(0),
            eps=self.eps
        ).item()
        return cos_sim, prompt_cos

    @staticmethod
    def _tokens(text):
        return set(re.findall(r"\b\w+\b", text))

    @classmethod
    def _lexical_overlap(cls, reference, response):
        ref_tokens, res_tokens = cls._tokens(reference), cls._tokens(response)
        if not ref_tokens or not res_tokens:
            return 0.0
        return len(ref_tokens & res_tokens) / len(ref_tokens | res_tokens)

    @staticmethod
    def _unsupported_number_ratio(reference, response):
        ref_numbers = set(re.findall(r"\b\d+\b", reference))
        resp_numbers = re.findall(r"\b\d+\b", response)
        if not resp_numbers:
            return 0.0
        unsupported = sum(1 for number in resp_numbers if number not in ref_numbers)
        return unsupported / len(resp_numbers)

    def build_features(self, cos_sim, prompt_cos, ref_text, res_clean, context_missing):
        """Stage 3: scalar feature extraction, ordering enforcement, and scaling."""
        ref_len = len(ref_text.split())
        res_len = len(res_clean.split())
        length_ratio = res_len / ref_len if ref_len > 0 else 0.0

        feature_dict = {
            "cosine_similarity": cos_sim,
            "prompt_response_cosine": prompt_cos,
            "reference_length": ref_len,
            "response_length": res_len,
            "length_ratio": length_ratio,
            "lexical_overlap": self._lexical_overlap(ref_text, res_clean),
            "unsupported_number_ratio": self._unsupported_number_ratio(ref_text, res_clean),
            "context_missing": context_missing,
        }

        missing = set(self.features) - set(feature_dict)
        if missing:
            raise KeyError(
                f"Metadata feature schema does not match build_features (missing: {sorted(missing)}). "
                "Re-run the feature-engineering and training cells to regenerate the artifacts."
            )

        feature_vector = np.array(
            [[feature_dict[col] for col in self.features]],
            dtype=np.float32
        )
        return self.scaler.transform(feature_vector)

    # ------------------------------------------------------------------
    # Classification
    # ------------------------------------------------------------------
    def classify_tier1(self, scaled_vector):
        """Tier-1 multi-model voting ensemble (includes the PyTorch ANN)."""
        with torch.inference_mode():
            probs = self.tier1_model.predict_proba(scaled_vector)[0]
        return float(probs[1])

    def classify_tier2(self, premise, hypothesis):
        """Tier-2 NLI verification with strict label-mapping validation."""
        inputs = self.tokenizer(
            premise,
            hypothesis,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(self.device)

        with torch.inference_mode():
            outputs = self.tier2_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[0]

        entail_idx, neutral_idx, contrad_idx = None, None, None
        for idx, label in self.id2label.items():
            if "entail" in label:
                entail_idx = idx
            elif "neutral" in label:
                neutral_idx = idx
            elif "contradict" in label:
                contrad_idx = idx

        if entail_idx is None:
            raise RuntimeError(f"Could not identify entailment label from id2label: {self.id2label}")

        entail_prob = float(probs[entail_idx].item())
        neutral_prob = float(probs[neutral_idx].item()) if neutral_idx is not None else 0.0
        contrad_prob = float(probs[contrad_idx].item()) if contrad_idx is not None else 0.0

        return {
            "contradiction": contrad_prob,
            "neutral": neutral_prob,
            "entailment": entail_prob,
            "hallucination_score": 1.0 - entail_prob,
        }

    def predict(self, context, prompt, response):
        """Cascaded evaluation using isolated per-tier thresholds."""
        ref_text_norm, res_clean_norm, context_missing = self.prepare_text(context, prompt, response)
        prompt_text_norm = self._normalize(prompt)

        cos_sim, prompt_cos = self.compute_embeddings(ref_text_norm, res_clean_norm, prompt_text_norm)
        scaled_vector = self.build_features(
            cos_sim, prompt_cos, ref_text_norm, res_clean_norm, context_missing
        )

        tier1_score = self.classify_tier1(scaled_vector)

        if not (self.gate_low <= tier1_score <= self.gate_high):
            return int(tier1_score >= self.threshold), tier1_score, "Tier-1 (Confident Fast-Path)"

        raw_premise = prompt if (pd.isna(context) or str(context).strip() == "") else context
        tier2_score = self.classify_tier2(raw_premise, response)["hallucination_score"]

        final_score = (self.tier1_weight * tier1_score) + (self.tier2_weight * tier2_score)
        return int(final_score >= self.cascade_threshold), final_score, "Tier-2 (Cascaded NLI Ensemble)"

    def predict_streaming(self, context, prompt, response, sentence_pattern=r"(?<=[.!?])\s+"):
        """Sentence-level scan for mid-generation / streaming hallucination detection.

        Scores every growing prefix of the response, so the first risky sentence
        can be flagged before generation finishes.
        """
        sentences = [s for s in re.split(sentence_pattern, str(response).strip()) if s.strip()]
        results, prefix = [], ""
        for i, sentence in enumerate(sentences, start=1):
            prefix = f"{prefix} {sentence}".strip()
            is_hall, score, path = self.predict(context, prompt, prefix)
            results.append({
                "sentence_index": i,
                "sentence": sentence,
                "risk_score": score,
                "path": path,
                "is_hallucination": is_hall,
            })
            if is_hall:
                break
        return results

In [ ]:
# NOTE: artifacts are already serialized at the end of the training cell.
# This cell is only a safety net for runtimes where that step was skipped and
# it no longer crashes with NameError when executed out of order.
import os
import json
import joblib

ARTIFACT_DIR = "./guardrail_artifacts"
metadata_path = os.path.join(ARTIFACT_DIR, "guardrail_metadata.json")
scaler_path = os.path.join(ARTIFACT_DIR, "guardrail_scaler.joblib")
ensemble_path = os.path.join(ARTIFACT_DIR, "tier1_voting_ensemble.joblib")

_required = ("deployment_metadata", "final_scaler", "tier1_final_model")
_missing = [name for name in _required if name not in globals()]

if _missing:
    print("Skipping serialization - run the training cell first. Missing:", ", ".join(_missing))
else:
    os.makedirs(ARTIFACT_DIR, exist_ok=True)

    print(f"Saving deployment metadata to: {metadata_path}")
    with open(metadata_path, 'w') as f:
        json.dump(deployment_metadata, f, indent=4)

    print(f"Saving StandardScaler to: {scaler_path}")
    joblib.dump(final_scaler, scaler_path)

    print(f"Saving Tier-1 Voting Ensemble model to: {ensemble_path}")
    joblib.dump(tier1_final_model, ensemble_path)

    print("Model artifacts saved successfully!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import json
import joblib
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sentence_transformers import SentenceTransformer

if 'CascadedGuardrailPipeline' not in globals():
    raise RuntimeError("Run the 'Cascaded Guardrail Pipeline Definition' cell before benchmarking.")

print("--- Step 8: Cascaded Ensemble Pipeline & Rigorous Latency Benchmark ---")

# --- Rigorous Latency Benchmark Runner ---
class BenchmarkTimer:
    """Robust wall-clock timer with explicit CUDA synchronization for mixed CPU/GPU pipelines."""
    def __init__(self, device_str):
        self.is_cuda = (device_str == "cuda")

    def start(self):
        if self.is_cuda:
            torch.cuda.synchronize()
        self.t0 = time.perf_counter()

    def stop(self):
        if self.is_cuda:
            torch.cuda.synchronize()
        return (time.perf_counter() - self.t0) * 1000


RANDOM_SEED = globals().get('RANDOM_SEED', 42)

benchmark_samples = (
    df[['context', 'prompt', 'response']]
    .sample(n=min(300, len(df)), random_state=RANDOM_SEED)
    .to_dict('records')
)

if len(benchmark_samples) < 15:
    raise ValueError(f"Need at least 15 test samples for benchmarking; found {len(benchmark_samples)}.")

def run_ensemble_benchmark(device_str):
    if device_str == "cuda" and not torch.cuda.is_available(): return

    print(f"\nEvaluating Cascaded Ensemble on Device: [{device_str.upper()}]")
    guardrail = CascadedGuardrailPipeline(ensemble_path, scaler_path, metadata_path, device_str=device_str)

    # 1. First-Inference Latency (Model already loaded)
    first_timer = BenchmarkTimer(device_str)
    first_timer.start()
    guardrail.predict(benchmark_samples[0]['context'], benchmark_samples[0]['prompt'], benchmark_samples[0]['response'])
    print(f"First-Inference Latency: {first_timer.stop():.2f} ms")

    # 2. Warm-up runs
    for row in benchmark_samples[1:10]:
        guardrail.predict(row['context'], row['prompt'], row['response'])

    # 3. Benchmark Execution
    timings = {'tier1_classifier': [], 'tier2_nli': [], 'e2e_total': []}
    tier2_trigger_count = 0

    for row in benchmark_samples[10:]:
        total_timer = BenchmarkTimer(device_str)
        total_timer.start()

        ref_text_norm, res_clean_norm, context_missing = guardrail.prepare_text(row['context'], row['prompt'], row['response'])
        prompt_text_norm = guardrail._normalize(row['prompt'])
        cos_sim, prompt_cos = guardrail.compute_embeddings(ref_text_norm, res_clean_norm, prompt_text_norm)
        scaled_vector = guardrail.build_features(cos_sim, prompt_cos, ref_text_norm, res_clean_norm, context_missing)

        # Measure Tier-1 Voting Classifier Only
        t1_timer = BenchmarkTimer(device_str)
        t1_timer.start()
        tier1_score = guardrail.classify_tier1(scaled_vector)
        timings['tier1_classifier'].append(t1_timer.stop())

        # Trigger Tier-2 NLI Fallback if score is within uncertainty gate
        if guardrail.gate_low <= tier1_score <= guardrail.gate_high:
            tier2_trigger_count += 1
            raw_premise = row['prompt'] if (pd.isna(row['context']) or str(row['context']).strip() == "") else row['context']
            t2_timer = BenchmarkTimer(device_str)
            t2_timer.start()
            _ = guardrail.classify_tier2(raw_premise, row['response'])
            timings['tier2_nli'].append(t2_timer.stop())

        timings['e2e_total'].append(total_timer.stop())

    total_evaluated = len(benchmark_samples[10:])
    print(f"Tier-2 Fallback Trigger Rate: {tier2_trigger_count / total_evaluated:.1%} ({tier2_trigger_count}/{total_evaluated} requests)")
    print(f"{'Pipeline Component / Stage':<34} | {'Mean (ms)':<9} | {'P95 (ms)':<9} | {'P99 (ms)':<9}")
    print("-" * 72)

    # Report Tier-1 Classifier Only
    t1_arr = timings['tier1_classifier']
    print(f"{'Tier-1 Classifier Only':<34} | {np.mean(t1_arr):<9.2f} | {np.percentile(t1_arr, 95):<9.2f} | {np.percentile(t1_arr, 99):<9.2f}")

    # Report Tier-2 NLI (Conditional on triggering)
    if timings['tier2_nli']:
        t2_arr = timings['tier2_nli']
        print(f"{'Tier-2 NLI (Triggered Requests)':<34} | {np.mean(t2_arr):<9.2f} | {np.percentile(t2_arr, 95):<9.2f} | {np.percentile(t2_arr, 99):<9.2f}")
    else:
        print(f"{'Tier-2 NLI (Triggered Requests)':<34} | {'N/A (0 triggers)':<9} | {'N/A':<9} | {'N/A':<9}")

    # Report Cascaded End-to-End
    e2e_arr = timings['e2e_total']
    print(f"{'Cascaded End-to-End':<34} | {np.mean(e2e_arr):<9.2f} | {np.percentile(e2e_arr, 95):<9.2f} | {np.percentile(e2e_arr, 99):<9.2f}")

run_ensemble_benchmark("cpu")
run_ensemble_benchmark("cuda")

In [ ]:
if 'CascadedGuardrailPipeline' not in globals():
    raise RuntimeError("Run the 'Cascaded Guardrail Pipeline Definition' cell first.")

# 1. Initialize the pipeline using your saved artifacts
guardrail = CascadedGuardrailPipeline(
    ensemble_file="./guardrail_artifacts/tier1_voting_ensemble.joblib",
    scaler_file="./guardrail_artifacts/guardrail_scaler.joblib",
    metadata_file="./guardrail_artifacts/guardrail_metadata.json",
    device_str="cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Define a helper function to test your own prompts cleanly
def check_hallucination(context, prompt, response):
    print(f"\nContext : {context}")
    print(f"Prompt  : {prompt}")
    print(f"Response: {response}")
    print("-" * 50)

    # Run the model!
    is_hallucinated, score, path = guardrail.predict(context, prompt, response)

    print(f"Risk Score : {score:.3f} (Thresholds: T1={guardrail.threshold:.3f}, T2={guardrail.cascade_threshold:.3f})")
    print(f"Path Taken : {path}")
    if is_hallucinated:
        print("DECISION: HALLUCINATION DETECTED")
    else:
        print("DECISION: SAFE (Factual/Supported)")

# --- Let's test it with some custom examples! ---

# Example A: A clean, factual response (Should be SAFE)
check_hallucination(
    context="The James Webb Space Telescope was launched on December 25, 2021.",
    prompt="When did the JWST launch?",
    response="It launched on December 25, 2021."
)

# Example B: A blatant hallucination (Should be DETECTED by Tier 1 fast-path)
check_hallucination(
    context="The James Webb Space Telescope was launched on December 25, 2021.",
    prompt="When did the JWST launch?",
    response="The JWST was launched in 1999 by the Soviet Union."
)

# Example C: A subtle hallucination/exaggeration (Might trigger the Tier 2 NLI fallback)
check_hallucination(
    context="The James Webb Space Telescope was launched on December 25, 2021. It orbits the Sun-Earth L2 point.",
    prompt="Where is the telescope located?",
    response="It orbits the Sun-Earth L2 point and is currently manned by three astronauts."
)

In [ ]:
# ---------------------------------------------------------------------------
# SUPERSEDED - intentionally disabled.
#
# This cell used to score the cascaded pipeline on the FULL dataframe (`df`),
# which silently leaks the training rows into the reported metrics. The
# unbiased evaluation lives in the "Final Evaluation on Strictly Held-Out
# Test Set" cell further down, which uses `df_test`.
#
# It is also very slow (~1 hour on 64k rows), so it is skipped on purpose.
# ---------------------------------------------------------------------------
print("Skipped: see the strict held-out evaluation cell below (uses df_test).")

In [ ]:
# @title
import os
import re
import json
import joblib
import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score
from tqdm import tqdm

if 'CascadedGuardrailPipeline' not in globals():
    raise RuntimeError("Run the 'Cascaded Guardrail Pipeline Definition' cell before evaluating.")

print("--- Final Evaluation on Strictly Held-Out Test Set ---")

# 2. Artifact Paths (Fallback definitions just in case they were lost from memory)
ARTIFACT_DIR = "./guardrail_artifacts"
metadata_path = os.path.join(ARTIFACT_DIR, "guardrail_metadata.json")
scaler_path = os.path.join(ARTIFACT_DIR, "guardrail_scaler.joblib")
ensemble_path = os.path.join(ARTIFACT_DIR, "tier1_voting_ensemble.joblib")

# 3. Initialize the deployment pipeline artifact
guardrail = CascadedGuardrailPipeline(
    ensemble_path,
    scaler_path,
    metadata_path,
    device_str="cuda" if torch.cuda.is_available() else "cpu"
)

# 4. Correctly target df_test
# IMPORTANT: If df_test throws a NameError here, you MUST run your train/test split cell higher up in the notebook!
test_records = df_test[['context', 'prompt', 'response']].to_dict('records')
y_true_test = df_test[TARGET_COLUMN].values

if len(test_records) == 0:
    raise ValueError("df_test is empty! Ensure your train/test split step ran correctly before evaluation.")

print(f"Evaluating {len(test_records)} held-out test samples through the Cascaded Pipeline...")

test_predictions = []
test_final_scores = []
paths_taken = {"Tier-1 (Fast-Path)": 0, "Tier-2 (NLI Cascade)": 0}

# 5. Process each test sample through the architecture
for row in tqdm(test_records):
    is_hallucinated, final_score, path = guardrail.predict(row['context'], row['prompt'], row['response'])

    test_predictions.append(is_hallucinated)
    test_final_scores.append(final_score)
    paths_taken[path] += 1

# 6. Calculate final metrics on unseen test data
test_ap = average_precision_score(y_true_test, test_final_scores)
test_trigger_rate = paths_taken["Tier-2 (NLI Cascade)"] / len(test_records)

print("\n=== UNBIASED HELD-OUT TEST PERFORMANCE ===")
print(f"Test Average Precision (AP) : {test_ap:.4f}")
print(f"Tier-2 Fallback Trigger Rate: {test_trigger_rate:.1%} ({paths_taken['Tier-2 (NLI Cascade)']} / {len(test_records)})")
print("-" * 45)

print("\nClassification Report:")
print(classification_report(y_true_test, test_predictions, target_names=["Safe (0)", "Hallucination (1)"], digits=4))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true_test, test_predictions)
print(f"True Negatives (Safe correctly identified)       : {cm[0][0]}")
print(f"False Positives (Safe flagged as hallucination)  : {cm[0][1]}")
print(f"False Negatives (Hallucinations missed)          : {cm[1][0]}")
print(f"True Positives  (Hallucinations caught)          : {cm[1][1]}")

print(f"\nRouting Breakdown:")
print(f" - Tier-1 Fast-Path Resolved : {paths_taken['Tier-1 (Fast-Path)']} ({paths_taken['Tier-1 (Fast-Path)']/len(test_records):.1%})")
print(f" - Tier-2 NLI Cascade Triggered: {paths_taken['Tier-2 (NLI Cascade)']} ({paths_taken['Tier-2 (NLI Cascade)']/len(test_records):.1%})")

In [ ]:
if 'guardrail' not in globals():
    raise RuntimeError(
        "`guardrail` is not initialized. Run the 'Cascaded Guardrail Pipeline Definition' "
        "cell and the 'Final Evaluation on Strictly Held-Out Test Set' cell first."
    )


def run_interactive_demo(context, prompt, response):
    print("\n" + "="*60)
    print("TESTING NEW INPUTS")
    print("="*60)
    print(f"Context : {context[:150]}..." if len(context) > 150 else f"📖 Context : {context}")
    print(f"Prompt  : {prompt}")
    print(f"Response: {response}")
    print("-" * 60)

    # Run prediction
    is_hallucinated, score, path = guardrail.predict(context, prompt, response)

    print(f"Risk Score     : {score:.4f}")
    print(f"Decision Path   : {path}")
    print(f"Thresholds Used : Tier-1 = {guardrail.threshold:.3f} | Tier-2 (Cascade) = {guardrail.cascade_threshold:.3f}")
    print("-" * 60)

    if is_hallucinated:
        print("RESULT: [HALLUCINATION DETECTED]")
    else:
        print("RESULT: [SAFE / FACTUAL]")
    print("="*60 + "\n")

# --- Test Cases ---
# Case 1: Strictly Factual
run_interactive_demo(
    context="The Eiffel Tower is located in Paris, France. It was completed in 1889.",
    prompt="Where is the Eiffel Tower and when was it built?",
    response="The Eiffel Tower is in Paris and was finished in 1889."
)

# Case 2: Hallucination
run_interactive_demo(
    context="The Eiffel Tower is located in Paris, France. It was completed in 1889.",
    prompt="Where is the Eiffel Tower and when was it built?",
    response="The Eiffel Tower is in Rome and was completed in 1950."
)


# --- Streaming / mid-generation demo (prefix-by-prefix scan) ---
stream_context = "The James Webb Space Telescope was launched on December 25, 2021."
stream_prompt = "Tell me about the JWST launch."
stream_response = (
    "The James Webb Space Telescope launched on December 25, 2021. "
    "It was built by NASA with contributions from ESA and CSA. "
    "The telescope is operated from the Moon by a crew of twelve astronauts."
)

print("\nStreaming scan - stops at the first risky sentence:")
for step in guardrail.predict_streaming(stream_context, stream_prompt, stream_response):
    verdict = "HALLUCINATION" if step["is_hallucination"] else "ok"
    print(f"  [{step['sentence_index']}] risk={step['risk_score']:.3f} {verdict} :: {step['sentence'][:90]}")


In [ ]:
# =====================================================================
# 1) Artifacts: export / import / reload + smoke test
# =====================================================================
# This cell lets you reuse the trained guardrail WITHOUT re-running training.
# Minimal fresh-session order:  torch -> ANN class -> pipeline class -> this cell.
import os
import json
import shutil
import zipfile
import joblib
import torch

DEFAULT_ARTIFACT_DIR = globals().get("ARTIFACT_DIR", "./guardrail_artifacts")
ARTIFACT_FILES = (
    "guardrail_metadata.json",
    "guardrail_scaler.joblib",
    "tier1_voting_ensemble.joblib",
)


def export_guardrail(artifact_dir=None, zip_name="guardrail_artifacts.zip", download=False):
    """Zip the trained artifacts so they can be downloaded and reloaded later."""
    artifact_dir = artifact_dir or DEFAULT_ARTIFACT_DIR
    missing = [f for f in ARTIFACT_FILES if not os.path.exists(os.path.join(artifact_dir, f))]
    if missing:
        raise FileNotFoundError(
            f"Missing artifacts in {artifact_dir}: {missing}. Run the training cell first."
        )

    archive = shutil.make_archive(zip_name.replace(".zip", ""), "zip", artifact_dir)
    print(f"Exported {archive} ({os.path.getsize(archive) / 1e6:.1f} MB)")

    if download:
        try:
            from google.colab import files
            files.download(archive)
        except ImportError:
            print("Not in Colab - the zip is already in the working directory.")
    return archive


def import_guardrail(archive_path=None, artifact_dir=None, upload=False):
    """Restore artifacts from a zip (Colab upload widget, or a local path)."""
    artifact_dir = artifact_dir or DEFAULT_ARTIFACT_DIR
    os.makedirs(artifact_dir, exist_ok=True)

    if upload:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError("upload=True only works in Colab. Pass archive_path instead.")
        for name in files.upload():
            with zipfile.ZipFile(name) as zf:
                zf.extractall(artifact_dir)
    else:
        if not archive_path or not os.path.exists(archive_path):
            raise FileNotFoundError(f"Archive not found: {archive_path}")
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(artifact_dir)

    print("Restored:", sorted(os.listdir(artifact_dir)))
    return artifact_dir


def load_guardrail(artifact_dir=None, device_str=None, activate=True, quiet=False):
    """Instantiate CascadedGuardrailPipeline from saved artifacts.

    Requires the ANN class cell and the pipeline class cell to have run
    (joblib needs TorchMLPClassifier to unpickle the ensemble).
    """
    artifact_dir = artifact_dir or DEFAULT_ARTIFACT_DIR

    if "TorchMLPClassifier" not in globals():
        raise RuntimeError(
            "Run the 'ANN: PyTorch multi-layer perceptron' cell first - "
            "joblib cannot unpickle the ensemble without TorchMLPClassifier."
        )
    if "CascadedGuardrailPipeline" not in globals():
        raise RuntimeError("Run the 'Cascaded Guardrail Pipeline Definition' cell first.")

    paths = {
        "ensemble": os.path.join(artifact_dir, "tier1_voting_ensemble.joblib"),
        "scaler": os.path.join(artifact_dir, "guardrail_scaler.joblib"),
        "metadata": os.path.join(artifact_dir, "guardrail_metadata.json"),
    }
    for label, path in paths.items():
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing {label}: {path}")

    if device_str is None:
        device_str = "cuda" if torch.cuda.is_available() else "cpu"

    pipe = CascadedGuardrailPipeline(
        paths["ensemble"], paths["scaler"], paths["metadata"], device_str=device_str
    )

    if activate:
        globals()["guardrail"] = pipe

    if not quiet:
        with open(paths["metadata"]) as f:
            meta = json.load(f)
        print(
            f"Loaded guardrail | schema {meta['artifact_schema_version']} | "
            f"{len(meta['feature_columns'])} features | device {device_str.upper()}"
        )
        print(f"Tier-1 members: {meta['tier1_config']['models']}")
    return pipe

In [ ]:
# =====================================================================
# 2) Ask-any-question helpers + wiring smoke test
# =====================================================================
# Works with or without the dataset in memory.

def check_and_report(context, prompt, response, truth=None):
    """Score one example and print a human-readable verdict. Returns a dict."""
    pipe = globals().get("guardrail") or load_guardrail()

    flag, score, path = pipe.predict(context, prompt, response)

    print("Context :", (context[:160] + "...") if context and len(str(context)) > 160 else (context or "(none)"))
    print("Prompt  :", prompt)
    print("Response:", response)
    print("-" * 60)
    print(f"Risk score : {score:.4f}")
    print(f"Path       : {path}")
    print("Verdict    :", "HALLUCINATION DETECTED" if flag else "SAFE / GROUNDED")
    if truth is not None:
        expected = "HALLUCINATION" if int(truth) else "SAFE"
        print("Ground truth:", expected, "->", "correct" if int(flag) == int(truth) else "MISSED")

    return {"is_hallucination": int(flag), "risk": score, "path": path}


def sample_example(task_type=None, hallucinated=None, seed=None):
    """Pull a real example out of the loaded dataset (needs `df` in memory)."""
    if "df" not in globals():
        raise RuntimeError("`df` is not in memory. Run the data + feature cells, or pass your own text.")

    sub = df
    if task_type is not None:
        sub = sub[sub["task_type"] == task_type]
    if hallucinated is not None:
        sub = sub[sub["is_hallucination"] == int(hallucinated)]
    if sub.empty:
        raise ValueError("No rows match that filter.")

    row = sub.sample(1, random_state=seed).iloc[0]
    return row["context"], row["prompt"], row["response"], int(row["is_hallucination"])


def ab_test_same_context(task_type="qa", seed=0):
    """Score BOTH twins of one paired example - the cleanest sanity check."""
    if "df" not in globals():
        raise RuntimeError("`df` is not in memory.")

    pool = df[df["task_type"] == task_type]
    group_id = pool.sample(1, random_state=seed)["group_id"].iloc[0]
    pair = df[df["group_id"] == group_id].sort_values("is_hallucination")

    print(f"=== Paired example (group {group_id}) ===")
    for _, row in pair.iterrows():
        check_and_report(row["context"], row["prompt"], row["response"], row["is_hallucination"])
        print()


SMOKE_CASES = [
    ("The James Webb Space Telescope was launched on December 25, 2021.",
     "When did the JWST launch?",
     "It launched on December 25, 2021.",
     0),
    ("The James Webb Space Telescope was launched on December 25, 2021.",
     "When did the JWST launch?",
     "The JWST was launched in 1999 by the Soviet Union.",
     1),
    ("The Eiffel Tower is located in Paris, France. It was completed in 1889.",
     "Where is the Eiffel Tower and when was it built?",
     "The Eiffel Tower is in Rome and was completed in 1950.",
     1),
]


def smoke_test(pipe=None, cases=None, verbose=True):
    """Verify the artifacts load and score sanely, then print reference cases.

    The assertions check WIRING (range, path names, determinism). The printed
    cases are reference expectations from the training run - a mismatch there
    is a model finding, not an infrastructure error.
    """
    pipe = pipe or globals().get("guardrail") or load_guardrail()
    cases = cases or SMOKE_CASES

    ctx, prompt, resp, _ = cases[0]
    _, score_a, path_a = pipe.predict(ctx, prompt, resp)
    _, score_b, _ = pipe.predict(ctx, prompt, resp)

    assert 0.0 <= score_a <= 1.0, f"risk score out of range: {score_a}"
    assert abs(score_a - score_b) < 1e-6, "pipeline is not deterministic"
    assert "Tier-1" in path_a or "Tier-2" in path_a, f"unexpected path label: {path_a}"

    print("Wiring OK: score in [0,1], deterministic, path labels valid.\n")

    hits = 0
    for i, (c, q, r, expected) in enumerate(cases, start=1):
        flag, score, path = pipe.predict(c, q, r)
        hits += int(flag) == int(expected)
        status = "match" if int(flag) == int(expected) else "MISMATCH"
        if verbose:
            print(f"[{i}] expected={'hallucination' if expected else 'safe':>13} | "
                  f"predicted={'hallucination' if flag else 'safe':>13} | "
                  f"risk={score:.3f} | {status}")
    print(f"\nReference cases matched: {hits}/{len(cases)}")
    return hits, len(cases)

In [ ]:
# =====================================================================
# 3) Interactive form (ipywidgets) - works inside the notebook
# =====================================================================
import ipywidgets as w
from IPython.display import display, clear_output

_ctx = w.Textarea(description="Context", layout=w.Layout(width="100%", height="90px"))
_q = w.Text(description="Question", layout=w.Layout(width="100%"))
_a = w.Textarea(description="Answer", layout=w.Layout(width="100%", height="90px"))
_out = w.Output()
_btn = w.Button(description="Check", button_style="primary")


def _on_check(_):
    with _out:
        clear_output()
        if not (_q.value or _a.value):
            print("Enter at least a question and an answer.")
            return
        check_and_report(_ctx.value, _q.value, _a.value)


_btn.on_click(_on_check)
display(w.VBox([_ctx, _q, _a, _btn, _out]))

In [ ]:
# =====================================================================
# 4) Gradio app - single check, sentence scan and simulated streaming
# =====================================================================
import time
import pandas as pd

try:
    import gradio as gr
except ImportError:
    gr = None
    print("Gradio not installed. Run: !pip install gradio")


def gradio_check(context, prompt, response):
    if not str(response).strip():
        return "-", "-", "-"
    pipe = globals().get("guardrail") or load_guardrail(quiet=True)
    flag, score, path = pipe.predict(context, prompt, response)
    return (
        "HALLUCINATION" if flag else "SAFE",
        f"{score:.3f}",
        path,
    )


def gradio_scan(context, prompt, response):
    if not str(response).strip():
        return pd.DataFrame(columns=["#", "sentence", "risk", "path"]), "-"
    pipe = globals().get("guardrail") or load_guardrail(quiet=True)
    rows = pipe.predict_streaming(context, prompt, response)
    table = pd.DataFrame([
        [r["sentence_index"], r["sentence"], round(r["risk_score"], 3), r["path"]]
        for r in rows
    ], columns=["#", "sentence", "risk", "path"])
    flagged = [r for r in rows if r["is_hallucination"]]
    if flagged:
        verdict = "HALLUCINATION first detected at sentence %d" % flagged[0]["sentence_index"]
    else:
        verdict = "all sentences clear"
    return table, verdict


def gradio_stream(context, prompt, response, delay=0.04):
    """Generator -> Gradio renders the partial states as they arrive."""
    pipe = globals().get("guardrail") or load_guardrail(quiet=True)
    shown = ""
    for token in str(response).split():
        shown = f"{shown} {token}".strip()
        flag, score, path = pipe.predict(context, prompt, shown)
        yield (
            shown,
            "HALLUCINATION" if flag else "safe",
            f"{score:.3f}",
            path,
        )
        time.sleep(delay)


if gr is not None:
    EXAMPLES = [
        ["The James Webb Space Telescope was launched on December 25, 2021.",
         "When did the JWST launch?",
         "It launched on December 25, 2021."],
        ["The James Webb Space Telescope was launched on December 25, 2021.",
         "When did the JWST launch?",
         "The JWST was launched in 1999 by the Soviet Union."],
        ["The JWST orbits the Sun-Earth L2 point.",
         "Where is the telescope?",
         "It orbits the Sun-Earth L2 point and is currently manned by three astronauts."],
    ]

    with gr.Blocks(title="Hallucination Guardrail") as demo:
        gr.Markdown(
            "## Hallucination Guardrail\n"
            "Two-tier cascade: fast ensemble -> NLI cross-encoder fallback."
        )

        with gr.Tab("Single check"):
            c1 = gr.Textbox(label="Context", lines=3)
            q1 = gr.Textbox(label="Question", lines=1)
            a1 = gr.Textbox(label="Model answer", lines=3)
            b1 = gr.Button("Check", variant="primary")
            out1 = [gr.Textbox(label="Verdict"), gr.Textbox(label="Risk"), gr.Textbox(label="Path")]
            b1.click(gradio_check, inputs=[c1, q1, a1], outputs=out1)
            gr.Examples(examples=EXAMPLES, inputs=[c1, q1, a1])

        with gr.Tab("Sentence scan"):
            c2 = gr.Textbox(label="Context", lines=3)
            q2 = gr.Textbox(label="Question", lines=1)
            a2 = gr.Textbox(label="Model answer", lines=4)
            b2 = gr.Button("Scan", variant="primary")
            table2 = gr.Dataframe(
                headers=["#", "sentence", "risk", "path"],
                datatype=["number", "str", "number", "str"],
                wrap=True,
            )
            verdict2 = gr.Textbox(label="Overall")
            b2.click(gradio_scan, inputs=[c2, q2, a2], outputs=[table2, verdict2])

        with gr.Tab("Live typing"):
            c3 = gr.Textbox(label="Context", lines=3)
            q3 = gr.Textbox(label="Question", lines=1)
            a3 = gr.Textbox(label="Model answer", lines=3)
            b3 = gr.Button("Simulate streaming", variant="primary")
            out3 = [
                gr.Textbox(label="Response so far", lines=4),
                gr.Textbox(label="Verdict"),
                gr.Textbox(label="Risk"),
                gr.Textbox(label="Path"),
            ]
            b3.click(gradio_stream, inputs=[c3, q3, a3], outputs=out3)

    demo.launch(share=True)

# =====================================================================
# 5) How to reuse this guardrail in a fresh session
# =====================================================================
# 1. Models on disk (3 files, all you need to persist):
#       guardrail_artifacts/
#         guardrail_metadata.json
#         guardrail_scaler.joblib
#         tier1_voting_ensemble.joblib
#    The HuggingFace models (all-MiniLM-L6-v2, nli-deberta-v3-small) are
#    re-downloaded automatically on first use and are NOT in the zip.
#
# 2. Run order in a fresh runtime - no dataset or retraining required:
#       [torch setup] -> [pip install sentence-transformers] ->
#       [ANN definition cell] -> [pipeline definition cell] ->
#       [artifacts cell (previous-but-two)] ->
#           guardrail = load_guardrail("./guardrail_artifacts")
#
# 3. GOTCHA: the ANN class cell MUST run before joblib.load, otherwise:
#       AttributeError: Can't get attribute 'TorchMLPClassifier'
#    (the ensemble pickles the class by reference, not by value)
#
# 4. Typical latency once warm: ~35 ms Tier-1, ~53 ms end-to-end on GPU,
#    ~230-500 ms on CPU. First call pays 3-8 s to load the 568 MB NLI model.
#
# 5. Persisting in Colab: point ARTIFACT_DIR at Drive in the training cell
#    AND in this loader, e.g.
#       ARTIFACT_DIR = "/content/drive/MyDrive/guardrail_artifacts"
#    so a runtime disconnect does not wipe the models.